---
title: Bathymetry visualization
subtitle: Geoprocessing with Python
venue: Pythia Cook-off 2026
---

## Overview

This notebook visualizes seabed bathymetry with the modern raster stack: `rioxarray` to load georeferenced data as a labelled array, `xarray-spatial` for terrain shading, and Matplotlib's `LightSource` for color-shaded relief. None of the hillshade math is written by hand. For an introduction to the underlying I/O, see {doc}`rasterio-intro`.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as mticker

# rioxarray registers the .rio accessor on xarray objects when imported
import rioxarray
from xrspatial import hillshade

# Tick formatter without scientific notation
formatter = mticker.FuncFormatter(lambda x, pos: f"{int(x):,}")

## Reading the data

We load the `gifford_bathy` raster with `rioxarray`. The result is an `xarray.DataArray` that already carries its coordinates, resolution, and CRS, so we never track the affine transform by hand. `masked=True` turns the nodata value into `NaN`, and `squeeze` drops the single-band dimension.

In [ ]:
data_path = Path("../data/gifford_bathy.tif")

bathy = rioxarray.open_rasterio(data_path, masked=True).squeeze()

print("dims       :", bathy.dims)
print("shape      :", bathy.shape)
print("CRS        :", bathy.rio.crs)
print("resolution :", bathy.rio.resolution())
print("depth range:", (float(bathy.min()), float(bathy.max())))
bathy

## A first look

Because the data is a labelled xarray object, a map-aware plot with a colorbar is a single method call. The axes are in projected map coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
bathy.plot(ax=ax, cmap="viridis", cbar_kwargs={"label": "Bathymetry (m)"})
ax.set_title("Gifford bathymetry")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.set_aspect("equal")
ax.xaxis.set_major_formatter(formatter)
ax.yaxis.set_major_formatter(formatter)
plt.show()

## Hillshade with xarray-spatial

A hillshade simulates how the surface would look lit by a low sun, which reveals subtle relief. `xarray-spatial` computes it directly from the DataArray using its built-in slope and aspect, so we compute no gradients ourselves. `azimuth` is the compass direction of the light and `angle_altitude` is its height above the horizon.

In [ ]:
shaded = hillshade(bathy, azimuth=315, angle_altitude=45)

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
shaded.plot(ax=ax, cmap="gray", add_colorbar=False)
ax.set_title("Gifford hillshade")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.set_aspect("equal")
ax.xaxis.set_major_formatter(formatter)
ax.yaxis.set_major_formatter(formatter)
plt.show()

## Color-shaded relief

The most informative view overlays the depth colormap on the shaded relief. Matplotlib's `LightSource.shade` blends a colormap with illumination in a single call. `vert_exag` exaggerates the vertical relief, useful for the gentle gradients of the seafloor. Because the blended image is an RGB array, we build a matching colorbar from a `ScalarMappable`.

In [ ]:
ls = LightSource(azdeg=315, altdeg=45)
dx, dy = bathy.rio.resolution()

rgb = ls.shade(
    bathy.values,
    cmap=plt.cm.viridis,
    vert_exag=5,
    dx=abs(dx),
    dy=abs(dy),
    blend_mode="soft",
)

# Map-coordinate extent for imshow: (left, right, bottom, top)
left, bottom, right, top = bathy.rio.bounds()

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
ax.imshow(rgb, extent=(left, right, bottom, top), origin="upper")

norm = Normalize(vmin=float(bathy.min()), vmax=float(bathy.max()))
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap="viridis"), ax=ax, shrink=0.85)
cbar.set_label("Bathymetry (m)")

ax.set_title("Gifford color-shaded relief")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.xaxis.set_major_formatter(formatter)
ax.yaxis.set_major_formatter(formatter)
plt.show()

:::{important} The two shading sections are alternatives
The `xarray-spatial` hillshade and this `LightSource` view are two separate ways to render relief, not a chain. `LightSource.shade` computes its own hillshade internally and blends it with the colormap, so the earlier `shaded` array is not an input here. Reach for `xarray-spatial`'s `hillshade` when you need the relief as reusable data, such as saving it as its own raster or scaling across large Dask-backed arrays; reach for `LightSource.shade` when you just want the finished color-and-relief figure.
:::

## A reusable function

The same three steps, load, shade, and plot, apply to any single-band bathymetry raster. We wrap them in one function and apply it to a second survey, Point Cloates.

In [ ]:
def plot_shaded_relief(path, title, azimuth=315, altitude=45, vert_exag=5, cmap="viridis"):
    """Load a single-band raster with rioxarray and plot color-shaded relief."""
    da = rioxarray.open_rasterio(path, masked=True).squeeze()

    ls = LightSource(azdeg=azimuth, altdeg=altitude)
    dx, dy = da.rio.resolution()
    rgb = ls.shade(
        da.values, cmap=plt.get_cmap(cmap), vert_exag=vert_exag,
        dx=abs(dx), dy=abs(dy), blend_mode="soft",
    )

    left, bottom, right, top = da.rio.bounds()
    fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
    ax.imshow(rgb, extent=(left, right, bottom, top), origin="upper")

    norm = Normalize(vmin=float(da.min()), vmax=float(da.max()))
    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.85)
    cbar.set_label("Bathymetry (m)")

    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)
    return da

## Applying to another survey

In [ ]:
pc_bathy = plot_shaded_relief("../data/pc_bathy.tif", "Point Cloates color-shaded relief")
plt.show()